In [2]:
import os
import glob
import pandas as pd

# Define path to logs
LOGS_DIR = '../../logs'

data = []

# Iterate over all directories in logs
for folder_name in os.listdir(LOGS_DIR):
    folder_path = os.path.join(LOGS_DIR, folder_name)
    
    if not os.path.isdir(folder_path):
        continue
        
    if 'test' not in folder_name:
        continue
        
    # Determine category
    if 'test-grasp' in folder_name:
        category = 'test-grasp'
    elif 'test-pp' in folder_name:
        category = 'test-pp'
    else:
        continue # Skip if neither
        
    # Determine split
    if 'unseen' in folder_name:
        split = 'unseen'
    else:
        split = 'seen'
        
    # Path to results
    results_dir = os.path.join(folder_path, 'results')
    if not os.path.exists(results_dir):
        continue
        
    # Parse txt files
    txt_files = glob.glob(os.path.join(results_dir, '*.txt'))
    
    for txt_file in txt_files:
        with open(txt_file, 'r') as f:
            line = f.read().strip()
            if not line:
                continue
            
            # Parse line
            # Format: prompt avg_success avg_step avg_success_step avg_reward
            # Prompt can contain spaces. The last 4 are numbers.
            parts = line.split()
            if len(parts) < 5:
                continue
                
            try:
                avg_reward = float(parts[-1])
                avg_success_step = float(parts[-2])
                avg_step = float(parts[-3])
                avg_success = float(parts[-4])
                
                prompt = " ".join(parts[:-4])
                
                data.append({
                    'Folder': folder_name,
                    'Category': category,
                    'Split': split,
                    'Prompt': prompt,
                    'Average Success Rate': avg_success,
                    'Average Success Step': avg_success_step
                })
            except ValueError:
                print(f"Error parsing file: {txt_file}")
                continue

df = pd.DataFrame(data)